<a href="https://colab.research.google.com/github/Psynotch/firstrep/blob/master/Session11_Independent_Reproducibility_Audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip -q install python-flint==0.9.0 mpmath==1.3.0

In [2]:
import csv
import json
import time
import hashlib
import platform
import importlib.metadata

import mpmath as mp

from fractions import Fraction
from flint import acb, arb, arb_series, ctx

print("SESSION 11: REPRODUCIBILITY AUDIT")
print("---------------------------------")
print("Python:", platform.python_version())
print("python-flint:", importlib.metadata.version("python-flint"))
print("mpmath:", importlib.metadata.version("mpmath"))

SESSION 11: REPRODUCIBILITY AUDIT
---------------------------------
Python: 3.13.18
python-flint: 0.9.0
mpmath: 1.3.0


In [4]:
T_LO_TEXT = "10"
T_HI_TEXT = "5000"

def compute_zero_set(bits):

    ctx.prec = bits

    start = time.time()

    n_lo_ball = arb(T_LO_TEXT).zeta_nzeros()
    n_hi_ball = arb(T_HI_TEXT).zeta_nzeros()

    n_lo = n_lo_ball.unique_fmpz()
    n_hi = n_hi_ball.unique_fmpz()

    if n_lo is None or n_hi is None:
        raise RuntimeError(
            f"Count was not isolated at {bits} bits."
        )

    n_lo = int(n_lo)
    n_hi = int(n_hi)

    number = n_hi - n_lo

    zeros = list(
        acb.zeta_zeros(n_lo + 1, number)
    )

    if len(zeros) != number:
        raise RuntimeError("Incorrect number of zero enclosures.")

    elapsed = time.time() - start

    return {
        "bits": bits,
        "n_lo": n_lo,
        "n_hi": n_hi,
        "count": number,
        "zeros": zeros,
        "seconds": elapsed
    }


result_192 = compute_zero_set(192)
result_256 = compute_zero_set(256)

for result in [result_192, result_256]:
    print(
        f"\nPrecision: {result['bits']} bits",
        f"\nN(10): {result['n_lo']}",
        f"\nN(5000): {result['n_hi']}",
        f"\nZeros: {result['count']}",
        f"\nElapsed: {result['seconds']:.2f} seconds"
    )


Precision: 192 bits 
N(10): 0 
N(5000): 4520 
Zeros: 4520 
Elapsed: 70.11 seconds

Precision: 256 bits 
N(10): 0 
N(5000): 4520 
Zeros: 4520 
Elapsed: 81.06 seconds


In [5]:
#Compare all 4,520 enclosures
print("CROSS-PRECISION COMPARISON")
print("--------------------------")

assert result_192["count"] == 4520
assert result_256["count"] == 4520

overlap_count = 0
nonoverlapping = []

for i, (z192, z256) in enumerate(
    zip(result_192["zeros"], result_256["zeros"]),
    start=1
):
    if z192.overlaps(z256):
        overlap_count += 1
    else:
        nonoverlapping.append(i)

print("Compared:", 4520)
print("Overlapping enclosures:", overlap_count)
print("Non-overlapping:", len(nonoverlapping))

if nonoverlapping:
    print("Problematic indices:", nonoverlapping[:20])

assert overlap_count == 4520

CROSS-PRECISION COMPARISON
--------------------------
Compared: 4520
Overlapping enclosures: 4520
Non-overlapping: 0


In [6]:
#Build certified intervals from the 256-bit results
ctx.prec = 256

def rational_ball(q):
    return arb(f"{q.numerator}/{q.denominator}")


zeros = result_256["zeros"]

centers = [
    Fraction(z.imag.mid().str(70, radius=False))
    for z in zeros
]

records = []

for i, zero in enumerate(zeros):

    center = centers[i]
    gamma = zero.imag

    radius = Fraction(1, 100)

    if i > 0:
        radius = min(
            radius,
            (center - centers[i - 1]) / 4
        )

    if i + 1 < len(centers):
        radius = min(
            radius,
            (centers[i + 1] - center) / 4
        )

    radius = min(
        radius,
        (center - Fraction(10)) / 4,
        (Fraction(5000) - center) / 4
    )

    if radius <= 0:
        raise RuntimeError(f"Invalid radius at zero {i+1}")

    left = center - radius
    right = center + radius

    contained = (
        rational_ball(left).upper() < gamma.lower()
        and gamma.upper() < rational_ball(right).lower()
    )

    if not contained:
        raise RuntimeError(
            f"Zero {i+1}: enclosure is not contained in interval."
        )

    records.append({
        "index": i + 1,
        "zero": zero,
        "gamma": gamma,
        "left": left,
        "right": right
    })


for i in range(len(records) - 1):
    assert records[i]["right"] < records[i + 1]["left"]

print("Certified intervals:", len(records))
print("All enclosures contained:", True)
print("All intervals disjoint:", True)

Certified intervals: 4520
All enclosures contained: True
All intervals disjoint: True


In [7]:
#Recheck all Hardy \(Z\) signs at 256 bits
def hardy_z(t_fraction):

    t = rational_ball(t_fraction)

    result = arb_series(
        [t, 1],
        prec=2
    ).riemann_siegel_z()

    return result.coeffs()[0]


def certified_sign(value):

    sign = value.sgn()

    if sign == arb(1):
        return "+"

    if sign == arb(-1):
        return "-"

    return "UNKNOWN"


verification_rows = []
failed = []

start = time.time()

for i, record in enumerate(records, start=1):

    zl = hardy_z(record["left"])
    zr = hardy_z(record["right"])

    sl = certified_sign(zl)
    sr = certified_sign(zr)

    passed = (
        sl != "UNKNOWN"
        and sr != "UNKNOWN"
        and sl != sr
    )

    if not passed:
        failed.append(record["index"])

    verification_rows.append({
        "zero_index": record["index"],
        "left": str(record["left"]),
        "right": str(record["right"]),
        "left_sign": sl,
        "right_sign": sr,
        "passed": passed
    })

    if i % 500 == 0 or i == len(records):
        print(
            f"Checked {i}/4520 | "
            f"Failed: {len(failed)}"
        )

verified = len(records) - len(failed)

print("\nVerified:", verified)
print("Failed:", len(failed))
print("Elapsed seconds:", round(time.time() - start, 2))

assert verified == 4520

Checked 500/4520 | Failed: 0
Checked 1000/4520 | Failed: 0
Checked 1500/4520 | Failed: 0
Checked 2000/4520 | Failed: 0
Checked 2500/4520 | Failed: 0
Checked 3000/4520 | Failed: 0
Checked 3500/4520 | Failed: 0
Checked 4000/4520 | Failed: 0
Checked 4500/4520 | Failed: 0
Checked 4520/4520 | Failed: 0

Verified: 4520
Failed: 0
Elapsed seconds: 26.57


In [8]:
#Cross-check selected signs with mpmath
mp.mp.dps = 80

sample_positions = [
    1, 2, 3, 100, 500, 1000, 1500,
    2000, 2260, 2500, 3000, 3500,
    4000, 4500, 4520
]

def mpmath_sign(t_fraction):

    t = (
        mp.mpf(t_fraction.numerator)
        / t_fraction.denominator
    )

    value = mp.siegelz(t)

    if value > 0:
        return "+"
    if value < 0:
        return "-"

    return "UNKNOWN"


mpmath_checks = []

for position in sample_positions:

    record = records[position - 1]
    arb_row = verification_rows[position - 1]

    mp_left = mpmath_sign(record["left"])
    mp_right = mpmath_sign(record["right"])

    agrees = (
        mp_left == arb_row["left_sign"]
        and mp_right == arb_row["right_sign"]
    )

    mpmath_checks.append({
        "position": position,
        "mpmath_left": mp_left,
        "arb_left": arb_row["left_sign"],
        "mpmath_right": mp_right,
        "arb_right": arb_row["right_sign"],
        "agrees": agrees
    })

    print(
        f"Zero {position}: "
        f"left {mp_left}/{arb_row['left_sign']}, "
        f"right {mp_right}/{arb_row['right_sign']}, "
        f"AGREE={agrees}"
    )

mpmath_agreements = sum(x["agrees"] for x in mpmath_checks)

print("\nIndependent numerical checks:",
      mpmath_agreements, "/", len(sample_positions))

Zero 1: left -/-, right +/+, AGREE=True
Zero 2: left +/+, right -/-, AGREE=True
Zero 3: left -/-, right +/+, AGREE=True
Zero 100: left +/+, right -/-, AGREE=True
Zero 500: left +/+, right -/-, AGREE=True
Zero 1000: left +/+, right -/-, AGREE=True
Zero 1500: left +/+, right -/-, AGREE=True
Zero 2000: left +/+, right -/-, AGREE=True
Zero 2260: left +/+, right -/-, AGREE=True
Zero 2500: left +/+, right -/-, AGREE=True
Zero 3000: left +/+, right -/-, AGREE=True
Zero 3500: left +/+, right -/-, AGREE=True
Zero 4000: left +/+, right -/-, AGREE=True
Zero 4500: left +/+, right -/-, AGREE=True
Zero 4520: left +/+, right -/-, AGREE=True

Independent numerical checks: 15 / 15


In [9]:
with open(
    "Session11_full_reproducibility_certificate.csv",
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "zero_index",
            "complex_zero_enclosure",
            "ordinate_enclosure",
            "left",
            "right",
            "left_sign",
            "right_sign",
            "passed"
        ]
    )

    writer.writeheader()

    for record, check in zip(records, verification_rows):

        writer.writerow({
            "zero_index": record["index"],
            "complex_zero_enclosure": str(record["zero"]),
            "ordinate_enclosure": str(record["gamma"]),
            "left": str(record["left"]),
            "right": str(record["right"]),
            "left_sign": check["left_sign"],
            "right_sign": check["right_sign"],
            "passed": check["passed"]
        })


csv_path = "Session11_full_reproducibility_certificate.csv"

with open(csv_path, "rb") as f:
    csv_hash = hashlib.sha256(f.read()).hexdigest()


# Read the file back and validate it.
with open(csv_path, newline="", encoding="utf-8") as f:
    saved_rows = list(csv.DictReader(f))

csv_valid = (
    len(saved_rows) == 4520
    and all(row["passed"] == "True" for row in saved_rows)
)

summary = {
    "session": 11,
    "region": {
        "real_part": [0, 1],
        "imaginary_part": [10, 5000]
    },
    "precisions_bits": [192, 256],
    "zero_count_192": result_192["count"],
    "zero_count_256": result_256["count"],
    "cross_precision_overlaps": overlap_count,
    "hardy_sign_checks": verified,
    "failed_intervals": failed,
    "mpmath_sample_checks": len(sample_positions),
    "mpmath_sample_agreements": mpmath_agreements,
    "csv_rows_after_reload": len(saved_rows),
    "csv_valid": csv_valid,
    "csv_sha256": csv_hash
}

with open(
    "Session11_reproducibility_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=2)

with open("Session11_reproducibility_summary.json", "rb") as f:
    json_hash = hashlib.sha256(f.read()).hexdigest()

print("CSV SHA-256:", csv_hash)
print("JSON SHA-256:", json_hash)
print("\n", json.dumps(summary, indent=2))

CSV SHA-256: 8274ea190d7dbc66fcc79ca1b03e83e1ef229de73ab49f64a50b190832a9bf23
JSON SHA-256: 0137e98336a4cf543eefdbcd71417df8bf4f1f813f03fdae86182212e46d2125

 {
  "session": 11,
  "region": {
    "real_part": [
      0,
      1
    ],
    "imaginary_part": [
      10,
      5000
    ]
  },
  "precisions_bits": [
    192,
    256
  ],
  "zero_count_192": 4520,
  "zero_count_256": 4520,
  "cross_precision_overlaps": 4520,
  "hardy_sign_checks": 4520,
  "failed_intervals": [],
  "mpmath_sample_checks": 15,
  "mpmath_sample_agreements": 15,
  "csv_rows_after_reload": 4520,
  "csv_valid": true,
  "csv_sha256": "8274ea190d7dbc66fcc79ca1b03e83e1ef229de73ab49f64a50b190832a9bf23"
}


In [10]:
final_success = (
    result_192["count"] == 4520
    and result_256["count"] == 4520
    and overlap_count == 4520
    and verified == 4520
    and len(failed) == 0
    and mpmath_agreements == len(sample_positions)
    and csv_valid
)

print("SESSION 11 FINAL AUDIT")
print("======================")

print("192-bit count:", result_192["count"])
print("256-bit count:", result_256["count"])
print("Matching enclosures:", overlap_count)
print("Hardy Z checks:", verified)
print("Failed intervals:", len(failed))
print("mpmath sample agreements:", mpmath_agreements)
print("CSV integrity:", csv_valid)

if final_success:
    print("\nSESSION 11: SUCCESS")
    print("Reproducibility checks passed.")
else:
    print("\nSESSION 11: NOT YET COMPLETE")
    print("Inspect the failed checks.")

SESSION 11 FINAL AUDIT
192-bit count: 4520
256-bit count: 4520
Matching enclosures: 4520
Hardy Z checks: 4520
Failed intervals: 0
mpmath sample agreements: 15
CSV integrity: True

SESSION 11: SUCCESS
Reproducibility checks passed.
